In [ ]:
#| default_exp sampling_utils

In [14]:
#| export
import sys
sys.path.append("..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from ddpm_backtest.data_loaders import get_nifty_regime_data,compute_regime_score,NiftyRiskDataset,fit_garch_volatility,prepare_dataloaders
from ddpm_backtest.noising_time import noisify,timestep_embedding,set_seed,split_context,cosine_beta_scheduler
from ddpm_backtest.models import FiLMLayer,f_net
from ddpm_backtest.diffusion_utils import TabularDDPM,EMA
from scipy.stats import rankdata
from scipy.interpolate import interp1d


In [3]:
risk_df = get_nifty_regime_data()
risk_df['regime_score'] = compute_regime_score(risk_df)
risk_df['regime_score'] = risk_df['regime_score'].fillna(0.5)
risk_df['regime'] = (risk_df['regime_score'] > 0.5).astype(int)
risk_df.drop(columns=['sma_200','Close'], inplace=True)
risk_df = fit_garch_volatility(risk_df)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Fitting GARCH(1,1) on training data...
                     Constant Mean - GARCH Model Results                      
Dep. Variable:          target_return   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -2619.48
Distribution:                  Normal   AIC:                           5246.95
Method:            Maximum Likelihood   BIC:                           5269.46
                                        No. Observations:                 2052
Date:                Sun, Apr 26 2026   Df Residuals:                     2051
Time:                        14:31:28   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.0776  1.

In [4]:
PAST_RETURN_COLS   = ([f"return_lag_{l}" for l in [1, 2, 3, 5, 10, 21]] +
                      [f"cum_return_{w}d" for w in [5, 10, 21, 63]])
RVOL_COLS          = ([f"rvol_{w}d" for w in [5, 10, 21, 63]] +
                      ["vol_ratio_5_21", "vol_ratio_21_63"])
ORIGINAL_CONT_COLS = ["realized_vol", "VIX", "RSI", "skew",
                       "kurtosis", "vol_spike", "RSI_velocity", "garch_vol"]
UNSCALED_CONT_COLS = ["regime_score",
                       "dow_sin", "dow_cos", "month_sin", "month_cos",
                       "dom_sin", "dom_cos", "quarter_sin", "quarter_cos"]
SCALED_CONT_COLS   = ORIGINAL_CONT_COLS + PAST_RETURN_COLS + RVOL_COLS

MARKET_DIM  = len(SCALED_CONT_COLS) + 1
TIME_DIM    = len(UNSCALED_CONT_COLS) - 1
CONTEXT_DIM = len(SCALED_CONT_COLS) + len(UNSCALED_CONT_COLS)
print(f"MARKET_DIM={MARKET_DIM}, TIME_DIM={TIME_DIM}, CONTEXT_DIM={CONTEXT_DIM}")

MARKET_DIM=25, TIME_DIM=8, CONTEXT_DIM=33


In [5]:
train_dl, val_dl, condition_df, scaler_x, scaler_cond, val_df = prepare_dataloaders(risk_df)
print(f"\nscaler_x mean={scaler_x.mean_[0]:.4f}  scale={scaler_x.scale_[0]:.4f}  (mean~0, scale~1)")

Train: 2,052  Val: 257  Test: 257

scaler_x mean=0.0591  scale=1.0063  (mean~0, scale~1)


In [6]:
T          = 100
betas      = cosine_beta_scheduler(T)
alphas     = 1 - betas
alphas_bar = torch.cumprod(torch.from_numpy(alphas).float(), dim=0)
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
alphas_bar = alphas_bar.to(device)
print(f"Device: {device} | T={T}")

Device: cpu | T=100


In [7]:
diffusion_model = TabularDDPM(
    d_in=1, cond_in_classes=2,
    scaled_cont_dim=len(SCALED_CONT_COLS),
    fixed_market_dim=1, time_dim=TIME_DIM,
    t_dim=64, dropout=0.1,
).to(device)
print(f"Parameters: {sum(p.numel() for p in diffusion_model.parameters()):,}")

optimizer        = torch.optim.Adam(diffusion_model.parameters(), lr=3e-4, weight_decay=1e-4)
num_epochs       = 500
p_uncond         = 0.15
warmup_eps       = 10
patience         = 25
best_val_mean    = np.inf
patience_counter = 0
ema_shadow_best  = None

def lr_lambda(epoch):
    if epoch < warmup_eps:
        return (epoch + 1) / warmup_eps
    progress = (epoch - warmup_eps) / max(1, num_epochs - warmup_eps)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
ema       = EMA(diffusion_model)

set_seed(42)
for epoch in range(num_epochs):
    diffusion_model.train()
    train_loss, n_train = 0.0, 0
    for x0, c, ctx in tqdm(train_dl, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
        x0, c, ctx = x0.to(device), c.to(device), ctx.to(device)
        if x0.dim() == 1: x0 = x0.unsqueeze(1)
        mc, tf     = split_context(ctx)
        drop_mask  = torch.rand(c.shape[0], device=device) < p_uncond
        xt, t, eps = noisify(T, x0, alphas_bar)
        eps_pred   = diffusion_model(xt, c, mc, tf, t.float(), drop_mask)
        snr     = alphas_bar[t] / (1.0 - alphas_bar[t])
        weights = (snr / snr.mean()).clamp(0.1, 10.0).unsqueeze(1)
        loss    = (weights * F.smooth_l1_loss(eps_pred, eps, reduction="none")).mean()
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(diffusion_model.parameters(), 1.0)
        optimizer.step(); ema.update(diffusion_model)
        train_loss += loss.item(); n_train += 1

    diffusion_model.eval()
    val_loss, n_val = 0.0, 0
    with torch.no_grad():
        for x0v, cv, ctxv in val_dl:
            x0v, cv, ctxv = x0v.to(device), cv.to(device), ctxv.to(device)
            if x0v.dim() == 1: x0v = x0v.unsqueeze(1)
            mcv, tfv = split_context(ctxv)
            xtv, tv, epsv = noisify(T, x0v, alphas_bar)
            val_loss += F.smooth_l1_loss(
                diffusion_model(xtv, cv, mcv, tfv, tv.float()), epsv).item()
            n_val += 1

    val_mean = val_loss / n_val
    if val_mean < best_val_mean:
        best_val_mean    = val_mean
        patience_counter = 0
        torch.save(diffusion_model.state_dict(), "best_model.pt")
        ema_shadow_best  = {k: v.clone() for k, v in ema.shadow.items()}
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    scheduler.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:03d} | Train {train_loss/n_train:.6f} | "
              f"Val {val_mean:.6f} | LR {scheduler.get_last_lr()[0]:.2e} | "
              f"Patience {patience_counter}/{patience}")

diffusion_model.load_state_dict(torch.load("best_model.pt"))
ema.shadow = ema_shadow_best
print(f"\nLoaded best model (val={best_val_mean:.6f})")

Parameters: 117,441


Epoch 010 | Train 0.244965 | Val 0.397826 | LR 3.00e-04 | Patience 5/25


Epoch 020 | Train 0.314466 | Val 0.327645 | LR 3.00e-04 | Patience 2/25


Epoch 030 | Train 0.266321 | Val 0.445400 | LR 2.99e-04 | Patience 1/25


Epoch 040 | Train 0.323422 | Val 0.586097 | LR 2.97e-04 | Patience 2/25


...

Epoch 110 | Train 0.260074 | Val 0.174288 | LR 2.70e-04 | Patience 13/25


Epoch 120 | Train 0.249470 | Val 0.182827 | LR 2.64e-04 | Patience 2/25


Epoch 130 | Train 0.294546 | Val 0.139505 | LR 2.58e-04 | Patience 12/25


Epoch 140 | Train 0.232505 | Val 0.510204 | LR 2.51e-04 | Patience 22/25


Early stopping at epoch 143

Loaded best model (val=0.134639)


In [ ]:
def _reverse_diffusion(model, xt, c_cond, market_cont, time_feats,
                        alphas_t, betas_t, alphas_bar, T,
                        guidance_scale, temperature, device):
    n      = xt.shape[0]
    d_all  = torch.ones(n,  dtype=torch.bool, device=device)
    d_none = torch.zeros(n, dtype=torch.bool, device=device)

    with torch.no_grad():
        for t in reversed(range(T)):
            tf          = torch.full((n,), t, device=device, dtype=torch.float)
            alpha_t     = alphas_t[t].unsqueeze(0).unsqueeze(1)
            beta_t      = betas_t[t].unsqueeze(0).unsqueeze(1)
            alpha_bar_t = alphas_bar[t].unsqueeze(0).unsqueeze(1)
            sqrt_a      = torch.sqrt(alpha_t)
            sqrt_1m_ab  = torch.sqrt(1.0 - alpha_bar_t).clamp(min=1e-5)

            eps_u = model(xt, c_cond, market_cont, time_feats, tf, drop_mask=d_all)
            eps_c = model(xt, c_cond, market_cont, time_feats, tf, drop_mask=d_none)
            eps   = (eps_u + guidance_scale * (eps_c - eps_u)).clamp(-3.0, 3.0)
            mean  = (xt - (beta_t / sqrt_1m_ab) * eps) / sqrt_a

            if t > 0:
                xt = mean + temperature * torch.sqrt(beta_t) * torch.randn_like(xt)
            else:
                xt = mean

    return xt


def _build_market_feats(condition_row, n_samples, scaler_cond, device):
    """Build market_cont and time_feats tensors from a conditioning row."""
    scaled_part  = scaler_cond.transform(pd.DataFrame([condition_row])[SCALED_CONT_COLS])
    regime_score = np.array([[condition_row["regime_score"]]])
    market_np    = np.concatenate([scaled_part, regime_score], axis=1)
    market_cont  = torch.tensor(market_np, dtype=torch.float32, device=device
                                ).expand(n_samples, -1)
    time_np    = pd.DataFrame([condition_row])[UNSCALED_CONT_COLS[1:]].values
    time_feats = torch.tensor(time_np, dtype=torch.float32, device=device
                              ).expand(n_samples, -1)
    c_cond = torch.full((n_samples,), int(condition_row["regime"]) + 1,
                         device=device, dtype=torch.long)
    return market_cont, time_feats, c_cond


def _sample_raw(condition_row, n_samples, model, alphas, betas,
                scaler_cond, guidance_scale, alphas_bar_in, temperature, device):
    """Run reverse diffusion. Returns raw tensor in SCALED residual space."""
    alphas_t = torch.from_numpy(alphas).float().to(device)
    betas_t  = torch.from_numpy(betas).float().to(device)
    _ab      = (alphas_bar_in if alphas_bar_in is not None else alphas_bar).to(device)
    model.eval(); model.to(device)
    mc, tf, c = _build_market_feats(condition_row, n_samples, scaler_cond, device)
    xt = torch.zeros((n_samples, 1), device=device)
    torch.nn.init.trunc_normal_(xt, mean=0, std=1.0, a=-3.0, b=3.0)
    return _reverse_diffusion(model, xt, c, mc, tf,
                               alphas_t, betas_t, _ab, T,
                               guidance_scale, temperature, device)


def calibrate_rescale_factor(condition_df, model, alphas, betas, scaler_x,
                               scaler_cond, alphas_bar_in, risk_df,
                               train_size=0.80, n_calib=2000,
                               guidance_scale=1.5, temperature=1.0):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_end    = int(train_size * len(risk_df))
    train_resids = risk_df.iloc[:train_end]["garch_resid"]

    target_5th   = train_resids.quantile(0.05)
    target_1st   = train_resids.quantile(0.01)
    target_std   = train_resids.std()

    mid_row = condition_df.iloc[len(condition_df) // 2]
    set_seed(0)
    xt = _sample_raw(mid_row, n_calib, model, alphas, betas,
                      scaler_cond, guidance_scale, alphas_bar_in, temperature, device)
    raw_resids    = scaler_x.inverse_transform(xt.cpu().numpy()).flatten()
    generated_5th = np.percentile(raw_resids, 5)
    generated_1st = np.percentile(raw_resids, 1)
    mu            = raw_resids.mean()
    factor_95 = (target_5th - mu) / (generated_5th - mu)
    factor_99 = (target_1st - mu) / (generated_1st - mu)
    factor = 0.4 * factor_95 + 0.6 * factor_99
    factor = float(np.clip(factor, 0.1, 10.0))

    print(f"5th target={target_5th:.4f}  generated={generated_5th:.4f}  factor_95={factor_95:.4f}")
    print(f"1st target={target_1st:.4f}  generated={generated_1st:.4f}  factor_99={factor_99:.4f}")
    print(f"Blended factor ({0.4:.0%} VaR95 + {0.6:.0%} VaR99): {factor:.4f}")
    return factor, target_std


'def sample_residuals(condition_row, n_samples, model, alphas, betas,\n                      scaler_x, scaler_cond, rescale_factor=1.0,\n                      guidance_scale=1.5, alphas_bar_in=None, temperature=1.0):\n    "\n    Generate STANDARDIZED RESIDUALS for one conditioning row.\n    rescale_factor maps generated std → actual training residual std.\n    Caller multiplies by garch_vol to get actual returns.\n    "\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    xt     = _sample_raw(condition_row, n_samples, model, alphas, betas,\n                          scaler_cond, guidance_scale, alphas_bar_in, temperature, device)\n    raw    = scaler_x.inverse_transform(xt.cpu().numpy()).flatten()\n    mu     = raw.mean()\n    return mu + (raw - mu) * rescale_factor'

In [11]:
#| export
def build_quantile_map(risk_df, train_size=0.80, n_quantiles=10000):
    train_end    = int(train_size * len(risk_df))
    train_resids = risk_df.iloc[:train_end]["garch_resid"].values
    quantile_levels = np.linspace(0.0001, 0.9999, n_quantiles)
    quantile_values = np.quantile(train_resids, quantile_levels)

    return interp1d(
        quantile_levels,
        quantile_values,
        bounds_error=False,
        fill_value=(quantile_values[0], quantile_values[-1])
    )

def apply_quantile_map(raw_resids, quantile_map):
    n     = len(raw_resids)
    ranks = (rankdata(raw_resids) - 0.5) / n
    return quantile_map(ranks)
quantile_map = build_quantile_map(risk_df)

def sample_residuals(condition_row, n_samples, model, alphas, betas,
                      scaler_x, scaler_cond, quantile_map=None,
                      guidance_scale=1.5, alphas_bar_in=None, temperature=1.0):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    xt     = _sample_raw(condition_row, n_samples, model, alphas, betas,
                          scaler_cond, guidance_scale, alphas_bar_in, temperature, device)
    raw    = scaler_x.inverse_transform(xt.cpu().numpy()).flatten()

    if quantile_map is not None:
        return apply_quantile_map(raw, quantile_map)
    return raw

In [ ]:
#| export
def generate_path_ensemble_garch(condition_df, n_paths, model, alphas, betas,
                                  scaler_x, scaler_cond, quantile_map=None,
                                  guidance_scale=1.5, temperature=1.0,
                                  alphas_bar_in=None):
    T_eval = len(condition_df)
    paths  = np.zeros((T_eval, n_paths))

    for i, (_, row) in enumerate(tqdm(condition_df.iterrows(),
                                       desc="GARCH-DDPM ensemble", total=T_eval)):
        residuals = sample_residuals(
            row, n_paths, model, alphas, betas, scaler_x, scaler_cond,
            guidance_scale=guidance_scale,quantile_map=quantile_map,alphas_bar_in=alphas_bar_in,
            temperature=temperature,
        )
        paths[i] = residuals * row["garch_vol"]

    actual = condition_df["target_return"].values
    var_95 = np.percentile(paths, 5,  axis=1)
    var_99 = np.percentile(paths, 1,  axis=1)
    es_95  = np.array([
        paths[t][paths[t] < var_95[t]].mean() if (paths[t] < var_95[t]).any() else var_95[t]
        for t in range(T_eval)
    ])
    es_99  = np.array([
        paths[t][paths[t] < var_99[t]].mean() if (paths[t] < var_99[t]).any() else var_99[t]
        for t in range(T_eval)
    ])
    print(f"\nGARCH-DDPM Ensemble ({T_eval} steps, {n_paths} paths):")
    print(f"  VaR 95% breach rate: {(actual < var_95).mean():.3f}  (target 0.050)")
    print(f"  VaR 99% breach rate: {(actual < var_99).mean():.3f}  (target 0.010)")
    return paths, var_95, var_99, es_95, es_99

In [13]:
set_seed(42)
ema.apply(diffusion_model)
rescale_factor, target_resid_std = calibrate_rescale_factor(
    condition_df, diffusion_model, alphas, betas,
    scaler_x, scaler_cond, alphas_bar,
    risk_df   = risk_df,         
    n_calib   = 2000,
    guidance_scale = 1.5,
    temperature    = 1.0,
)
ema.restore(diffusion_model)
set_seed(42)
ema.apply(diffusion_model)
test_resids = sample_residuals(
    condition_df.iloc[-1], 1000, diffusion_model, alphas, betas,
    scaler_x, scaler_cond,quantile_map = quantile_map,
    guidance_scale=1.5, alphas_bar_in=alphas_bar,
)
ema.restore(diffusion_model)

gv = condition_df.iloc[-1]["garch_vol"]
print(f"\nAfter rescaling:")
print(f"  resid std={test_resids.std():.4f}  (target {target_resid_std:.4f})")
print(f"  VaR95 generated: {np.percentile(test_resids * gv, 5):.5f}")
print(f"  VaR95 actual:    {np.percentile(condition_df['target_return'], 5):.5f}")

5th target=-1.6228  generated=-2095.6758  factor_95=-0.0068
1st target=-2.8248  generated=-3065.9622  factor_99=-0.0043
Blended factor (40% VaR95 + 60% VaR99): 0.1000

After rescaling:
  resid std=0.9985  (target 1.0065)
  VaR95 generated: -0.02594
  VaR95 actual:    -0.01391


In [ ]:
#| export
def evaluate_predictive_risk(generated_paths, actual_returns,
                              confidence_levels=[0.95, 0.99]):
    T_days = len(actual_returns)
    print(f"\n{'='*65}")
    print(f" PREDICTIVE VaR BACKTEST  (T={T_days} out-of-sample days)")
    print(f"{'='*65}")
    print(f"  {'Metric':<10} {'Target':>8} {'Actual':>8} {'Breaches':>10} {'p-val':>8} {'Status':>8}")
    print("  " + "-" * 56)
    results = {}
    for cl in confidence_levels:
        alpha      = 1 - cl
        pred_var   = np.percentile(generated_paths, alpha * 100, axis=1)
        is_breach  = actual_returns < pred_var
        n_breaches = is_breach.sum()
        actual_rate = n_breaches / T_days
        test   = binomtest(k=n_breaches, n=T_days, p=alpha, alternative="two-sided")
        p_val  = test.pvalue
        status = "PASS ✓" if p_val >= 0.05 else "FAIL ✗"
        print(f"  VaR {cl:.0%}   {alpha:>8.3%} {actual_rate:>8.3%} "
              f"{n_breaches:>4} / {T_days:<5} {p_val:>8.4f} {status:>8}")
        results[f"var_{cl}"]      = pred_var
        results[f"breaches_{cl}"] = is_breach
    return results


def plot_pit_histogram(generated_paths, actual_returns, bins=20):
    T_days     = len(actual_returns)
    pit_values = np.array([(generated_paths[t] <= actual_returns[t]).mean()
                            for t in range(T_days)])
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].hist(pit_values, bins=bins, density=True,
                 color="teal", edgecolor="black", alpha=0.7)
    axes[0].axhline(1.0, color="red", ls="--", lw=2, label="Ideal (Uniform)")
    axes[0].set_title("PIT Histogram"); axes[0].set_xlabel("Quantile")
    axes[0].set_ylabel("Density"); axes[0].legend(); axes[0].grid(alpha=0.3)
    counts, edges = np.histogram(pit_values, bins=bins, density=True)
    centers = (edges[:-1] + edges[1:]) / 2
    axes[1].bar(centers, counts - 1.0, width=edges[1]-edges[0],
                color=["tomato" if v > 0 else "steelblue" for v in counts - 1.0],
                edgecolor="black", alpha=0.8)
    axes[1].axhline(0, color="black", lw=1)
    axes[1].set_title("PIT Deviation from Uniform")
    axes[1].set_xlabel("Quantile"); axes[1].set_ylabel("Density - 1")
    axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    return pit_values


def plot_var_timeseries(actual_returns, var_dict, dates=None):
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    x = dates if dates is not None else np.arange(len(actual_returns))
    ax = axes[0]
    ax.plot(x, actual_returns, color="black", alpha=0.6, lw=1, label="Realized Return")
    colors = {"var_0.95": "orange", "var_0.99": "red"}
    styles = {"var_0.95": "--",     "var_0.99": "-."}
    for key in ["var_0.95", "var_0.99"]:
        if key in var_dict:
            ax.plot(x, var_dict[key], color=colors[key], ls=styles[key],
                    lw=1.5, label=f"Predicted {key.replace('var_','VaR ')}")
    if "breaches_0.99" in var_dict:
        bidx = np.where(var_dict["breaches_0.99"])[0]
        ax.scatter(x[bidx], actual_returns[bidx],
                   color="red", marker="x", s=80, zorder=5, label="99% Breach")
    ax.set_title("GARCH-DDPM Predictive VaR Backtest"); ax.set_ylabel("Log Return")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax = axes[1]
    for key, color, target in [("breaches_0.95","orange",0.05),
                                 ("breaches_0.99","red",0.01)]:
        if key in var_dict:
            roll = pd.Series(var_dict[key].astype(float)).rolling(63, min_periods=1).mean().values
            ax.plot(x, roll, color=color, label=f"63d breach rate ({key.split('_')[1]})")
            ax.axhline(target, color=color, ls="--", lw=1, alpha=0.6)
    ax.set_ylabel("Rolling Breach Rate"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

In [ ]:
set_seed(42)
actual_returns = condition_df["target_return"].dropna().values

ema.apply(diffusion_model)
paths, var_95, var_99, es_95, es_99 = generate_path_ensemble_garch(
    condition_df   = condition_df,
    n_paths        = 10000,
    model          = diffusion_model,
    alphas         = alphas,
    betas          = betas,
    scaler_x       = scaler_x,
    scaler_cond    = scaler_cond,
    quantile_map   = quantile_map,  
    guidance_scale = 1.0,
    temperature    = 0.9,
    alphas_bar_in  = alphas_bar,
)
ema.restore(diffusion_model)

var_results = evaluate_predictive_risk(paths, actual_returns)
pit_vals    = plot_pit_histogram(paths, actual_returns)
plot_var_timeseries(actual_returns, var_results, dates=condition_df.index.values)

GARCH-DDPM ensemble:   9%|▉         | 24/257 [12:45<2:06:55, 32.68s/it]